<a href="https://colab.research.google.com/github/afrahalbraq-eng/bayan-nlp-project/blob/main/Another_copy_of_Student_Lab1_Bilingual_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Clean the Messages | تنظيف الرسائل
**SDA-AIE-211 | Day 1 **

## القصة | The story
- 🇸🇦 وصلت إلى منصة **بيان** آلاف الرسائل من المواطنين، وفيها أرقام جوال وإيموجي وتطويل وأخطاء. قبل أي نموذج، يجب **تنظيف الرسائل بأمان** ثم **اختيار النموذج المناسب** للعربية والإنجليزية.
- 🇬🇧 The **Bayan** service received thousands of citizen messages full of phone numbers, emojis and noise. Before any model, we must **clean safely** and **pick the right model** for Arabic and English.

## الخطوات | Steps
| # | الخطوة | Step |
|---|---|---|
| 0 | تحميل البيانات | Load data |
| 1 | البحث عن العيوب | Find the defects |
| 2 | إخفاء البيانات الشخصية وتنظيف النص | Mask personal data and clean |
| 3 | تقسيم الجمل | Split sentences |
| 4 | مقارنة النماذج (Fertility) | Compare tokenizers |
| 5 | القرار وحفظ الملف | Decision and save |


In [ ]:
# ---- 0. Install libraries (run once per Colab session) ----
# datasets / accelerate / evaluate / seqeval -> later labs (training + evaluation)
# sentence-transformers / faiss-cpu          -> later labs (embeddings + similarity search)
# spacy                                      -> sentence segmentation in this lab
!pip -q install pandas numpy scikit-learn matplotlib transformers datasets accelerate evaluate seqeval sentence-transformers faiss-cpu spacy

## Step 0 — تحميل البيانات | Load data

In [ ]:
# ---- 1. Imports, Google Drive and paths ----
from google.colab import drive
drive.mount('/content/drive')

import re, json, unicodedata, random
from pathlib import Path
import numpy as np
import pandas as pd


# Change ONLY this path if your folder is somewhere else
DATA_DIR = Path('/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data')

# Fail early with a clear message if a file is missing
for name in ['bayan_feedback.csv', 'bayan_raw_sample.csv', 'pii_test_set.csv']:
    assert (DATA_DIR / name).exists(), f'Missing file: {DATA_DIR / name}'
print('All data files found.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All data files found.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---- 2. Load the main dataset ----
feedback = pd.read_csv(DATA_DIR / 'bayan_feedback.csv')
print(feedback.shape)
print(feedback.columns.tolist())

# Quick health check: missing values and language balance
print(feedback[['text', 'language']].isna().sum())
print(feedback['language'].value_counts())

(12000, 11)
['feedback_id', 'citizen_id', 'text', 'language', 'dialect_region', 'topic', 'sentiment', 'channel', 'city', 'service_entity', 'created_day']
text        0
language    0
dtype: int64
language
en    6000
ar    6000
Name: count, dtype: int64


## Step 1 — البحث عن العيوب | Find the defects
- 🇸🇦 اقرأ الرسائل الخام، ثم شاهد عدد كل عيب.
- 🇬🇧 Read the raw messages, then see how often each defect appears.

In [ ]:
# ---- 3. Look at raw records ----
raw = pd.read_csv(DATA_DIR / 'bayan_raw_sample.csv')
display(raw[['feedback_id', 'text', 'language']].head(12))

display(raw[['feedback_id','text','language']].head(12))
# TODO: add six defects you actually observed in the sample.

defects = pd.DataFrame([

    {
        "Defect": "Tatweel / Elongation",
        "Example": "التطبيــــــــق",
        "Description": "Unnecessary character elongation in Arabic text."
    },
    {
        "Defect": "Emoji",
        "Example": "الخدمة ممتازة 🥰",
        "Description": "Emojis used to express emotions or opinions."
    },
    {
        "Defect": "Mixed Script",
        "Example": "التطبيق لا يعمل after update",
        "Description": "Arabic and English words in the same sentence."
    },
    {
        "Defect": "PII-like Text",
        "Example": "0551234567",
        "Description": "Personal information such as phone numbers."
    },
    {
        "Defect": "Repeated Punctuation",
        "Example": "التطبيق لا يعمل!!!",
        "Description": "Repeated punctuation marks."
    },
    {
        "Defect": "Unicode Variation",
        "Example": "Different Unicode representations of Arabic characters",
        "Description": "Visually similar characters may have different Unicode encodings."
    }


])

display(defects)

,feedback_id,text,language
0,BF05091,مرة أخرى، أريد تغيير موعدي يوم الجمعة. شكرا مق...,ar
1,BF01164,I missed my connection because the schedule sh...,en
2,BF00918,تذكير: يا أخي ما وصلني المبلغ المسترد 200 ريال...,ar
3,BF05630,مرة أخرى؟ كل ما في الثلاجة يفسد ولم يعطنا أحد ...,ar
4,BF03709,Question: The trip took 61 minutes instead of ...,en
5,BF05073,خبر جيد: تم خصم 75 ريال مرتين من فاتورتي، ومن ...,ar
6,BF06579,The garbage was not collected today. This is n...,en
7,BF09052,Really frustrated. I need help with my reques...,en
8,BF00811,محبط فعلا. أريد تغيير موعدي يوم الاثنين. أـــر...,ar
9,BF06013,للتوثيق، يعمل على الحاسوب ومش بيشتغل على الجوا...,ar


,feedback_id,text,language
0,BF05091,مرة أخرى، أريد تغيير موعدي يوم الجمعة. شكرا مق...,ar
1,BF01164,I missed my connection because the schedule sh...,en
2,BF00918,تذكير: يا أخي ما وصلني المبلغ المسترد 200 ريال...,ar
3,BF05630,مرة أخرى؟ كل ما في الثلاجة يفسد ولم يعطنا أحد ...,ar
4,BF03709,Question: The trip took 61 minutes instead of ...,en
5,BF05073,خبر جيد: تم خصم 75 ريال مرتين من فاتورتي، ومن ...,ar
6,BF06579,The garbage was not collected today. This is n...,en
7,BF09052,Really frustrated. I need help with my reques...,en
8,BF00811,محبط فعلا. أريد تغيير موعدي يوم الاثنين. أـــر...,ar
9,BF06013,للتوثيق، يعمل على الحاسوب ومش بيشتغل على الجوا...,ar


,Defect,Example,Description
0,Tatweel / Elongation,التطبيــــــــق,Unnecessary character elongation in Arabic text.
1,Emoji,الخدمة ممتازة 🥰,Emojis used to express emotions or opinions.
2,Mixed Script,التطبيق لا يعمل after update,Arabic and English words in the same sentence.
3,PII-like Text,0551234567,Personal information such as phone numbers.
4,Repeated Punctuation,التطبيق لا يعمل!!!,Repeated punctuation marks.
5,Unicode Variation,Different Unicode representations of Arabic ch...,Visually similar characters may have different...


## Step 2 — إخفاء البيانات الشخصية وتنظيف النص | Mask personal data and clean
- 🇸🇦 **القاعدة الذهبية:** أخفِ البيانات الشخصية (PII) **أولًا**، ثم نظّف النص.
- 🇬🇧 **Golden rule:** mask personal data (PII) **first**, then clean the text.
- 🔒 الدالة `mask_pii` فيها تعابير نمطية (Regex) معقدة → **للقراءة فقط** /

Mask PII → Unicode NFC Normalization → Remove Tatweel → Normalize Whitespace → Strip Extra Spaces → Normalized Text  → Test (PII masking)

In [ ]:
# Version of the preprocessing pipeline
PREPROCESSING_VERSION = "bayan-preproc-v1"


# Function to hide personal information (PII)
def mask_pii(text):
    text = str(text)

    # Regular Expression for email addresses:
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL]', text
    )

    # Regular Expression for Saudi mobile numbers:
    text = re.sub(
        r'(?<!\d)05(?:[\s-]?\d){8}(?!\d)',
        '[PHONE]', text
    )

    # Regular Expression for Saudi national ID:
    text = re.sub(
        r'(?<!\d)[12]\d{9}(?!\d)',
        '[NATIONAL_ID]', text
    )

    return text


# Function to normalize Arabic and English text
def normalize(text):
    # Convert text to Unicode NFC format
    text = unicodedata.normalize('NFC', str(text))

    # Remove Arabic Tatweel (ـ)
    text = text.replace('ـ', '')

    # Replace multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces at the beginning and end
    return text.strip()


# Function to apply preprocessing in the correct order
def preprocess(text):
    # Step 1: Hide personal information FIRST
    text = mask_pii(text)

    # Step 2: Normalize the text
    text = normalize(text)

    return text


# Load the PII test dataset
pii = pd.read_csv(
    "/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/pii_test_set.csv"
)

# Test whether all expected PII values are masked
passed = [
    str(r.expected_value) not in mask_pii(r.text)  # check if the value available or not should return true
    for r in pii.itertuples()
]

# Display test results
print(f"PII masking tests: {sum(passed)}/{len(passed)}")

# Stop if any test fails
assert all(passed), "Masking test failed; check the regular expressions."


# TODO: Test phone numbers with different formats
test_numbers = [
    "0551234567",
    "(0551234567)",
    "055-123-4567",
    "055 123 4567"
]

for number in test_numbers:
    print(number, "→", mask_pii(number))

PII masking tests: 60/60
0551234567 → [PHONE]
(0551234567) → ([PHONE])
055-123-4567 → [PHONE]
055 123 4567 → [PHONE]


In [ ]:
feedback['text_clean'] = feedback['text'].map(preprocess)

## Step 3 — تقسيم الجمل | Segmentation
- 🇸🇦 أحيانًا تحتوي الرسالة على أكثر من جملة. نقسّمها عند علامات النهاية.
- 🇬🇧 A message can hold several sentences. We split at sentence-ending marks.

In [ ]:
def split_sentences(text):
    parts = re.split(r'(?<=[.!?؟])\s+', preprocess(text))   # نقسم بعد . ! ? ؟  \s أي مسافة بيضاء (Whitespace), + تكرار مرة واحدة او اكثر
    return [p for p in parts if p] #p موجودة وليست فارغة

print(split_sentences('الخدمة بطيئة جدا. هل يمكن المساعدة؟ شكرا'))

['الخدمة بطيئة جدا.', 'هل يمكن المساعدة؟', 'شكرا']


## Step 4 — مقارنة النماذج | Compare tokenizers
- 🇸🇦 **Fertility** = عدد القطع (tokens) ÷ عدد الكلمات. كلما قلّ الرقم كان النموذج "يفهم" اللغة بكفاءة أكبر.
- 🇬🇧 **Fertility** = tokens ÷ words. Lower means the model handles the language more efficiently.
- **p95:** طول 95٪ من الرسائل (نستخدمه لاحقًا لاختيار أقصى طول) / length that 95% of messages fit in. طول 95% من الرسائل بوحدة Tokens
- 🔒 الكود للقراءة فقط، شغّله وانظر الجدول / read-only — just run it and read the table.

In [ ]:
# استيراد AutoTokenizer لتحميل Tokenizer الخاص بكل نموذج
# Import AutoTokenizer to load each model's tokenizer

from transformers import AutoTokenizer


# تحديد النماذج التي نريد مقارنة Tokenizers الخاصة بها
# Define the models we want to compare

candidates = {
    'xlm-roberta-base': 'XLM-R',
    'bert-base-multilingual-cased': 'mBERT',
    'CAMeL-Lab/bert-base-arabic-camelbert-mix': 'CAMeLBERT',
}


# إنشاء قائمة فارغة لتخزين نتائج المقارنة
# Create an empty list to store comparison results

rows = []


# المرور على كل نموذج من النماذج الثلاثة
# Loop through each of the three models

for ckpt, label in candidates.items(): #name of the model and shortcut

    # تحميل Tokenizer الخاص بالنموذج الحالي
    # Load the tokenizer for the current model

    tok = AutoTokenizer.from_pretrained(ckpt)


    # اختبار النموذج على اللغتين العربية والإنجليزية
    # Test the tokenizer on Arabic and English

    for lang in ['ar', 'en']:

        # اختيار أول 300 رسالة من اللغة الحالية
        # Select the first 300 messages in the current language

        texts = feedback.loc[
            feedback.language == lang, 'text_clean'
        ].head(300).tolist()


        # حساب إجمالي عدد Tokens في الرسائل
        # Count the total number of tokens across all messages

        pieces = sum(len(tok.tokenize(t)) for t in texts) #tokens of the 300 message


        # حساب إجمالي عدد الكلمات في الرسائل
        # Count the total number of words across all messages

        words = sum(len(t.split()) for t in texts) #words in the 300 message


        # حساب عدد Tokens في كل رسالة
        # Calculate the number of tokens in each message

        lengths = [len(tok(t)['input_ids']) for t in texts] #from each token ID


        # إضافة نتائج النموذج واللغة إلى القائمة
        # Add the model and language results to the list

        rows.append({
            'model': label,
            'language': lang,

            # Fertility = إجمالي Tokens ÷ إجمالي الكلمات
            # Average number of tokens per word

            'fertility': round(pieces / words, 2), #round 2

            # حساب المئين الخامس والتسعين لطول الرسائل
            # Calculate the 95th percentile of message lengths

            'p95_length': int(np.percentile(lengths, 95))
        })


# تحويل قائمة النتائج إلى DataFrame لعرضها كجدول
# Convert the results list into a DataFrame

audit = pd.DataFrame(rows)


# عرض جدول المقارنة
# Display the comparison table

display(audit)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

,model,language,fertility,p95_length
0,XLM-R,ar,1.71,41
1,XLM-R,en,1.37,41
2,mBERT,ar,2.29,56
3,mBERT,en,1.41,42
4,CAMeLBERT,ar,1.44,35
5,CAMeLBERT,en,2.51,73


## Step 5 — القرار وحفظ الملف | Decision and save
- 🇸🇦 اختر **نموذجًا للعربية** و**نموذجًا للغتين**، واكتب سببك مستندًا إلى جدول Step 4.
- 🇬🇧 Choose an **Arabic model** and a **bilingual model**, and justify with the Step 4 table.

In [ ]:
# TODO: Choose checkpoint(s) using your measured fertility,
# p95 length, and language coverage.

decision = {
    'arabic_checkpoint': 'CAMeL-Lab/bert-base-arabic-camelbert-mix',
    'bilingual_checkpoint': 'xlm-roberta-base',
    'evidence': 'CAMeLBERT showed the lowest fertility for Arabic text, making it highly efficient, while xlm-roberta-base provided optimal performance for bilingual coverage.'
}

# Print the decision as formatted JSON
print(json.dumps(decision, indent=2))

# حفظ الملفات للابات القادمة | Save files for the next labs
import os
os.makedirs('/content/artifacts', exist_ok=True)
json.dump(decision, open('/content/artifacts/lab1_decision.json', 'w'), ensure_ascii=False, indent=2)
feedback.to_csv('/content/bayan_feedback_clean.csv', index=False)
print('✅ Saved decision + clean data')

{
  "arabic_checkpoint": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
  "bilingual_checkpoint": "xlm-roberta-base",
  "evidence": "CAMeLBERT showed the lowest fertility for Arabic text, making it highly efficient, while xlm-roberta-base provided optimal performance for bilingual coverage."
}
✅ Saved decision + clean data


In [ ]:
from google.colab import files
files.download('/content/bayan_feedback_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Thinking challenges
1. Why preserve emoji for sentiment?
2. What breaks if train and serving use different normalisers?
3. Defend one multilingual tokenizer versus separate Arabic/English models using your audit.